# Sistema de Detección de Presencia por Fuerza Mínima (RH56DFTP-2R)
Este script independiente transforma los sensores de fuerza internos de la mano robótica Inspire RH56DFTP-2R en sensores táctiles de presencia. En lugar de utilizar el límite de fuerza por hardware como un tope de presión bruta, el programa monitorea en tiempo real las variaciones de carga a alta frecuencia ($\sim 50\text{ Hz}$) y detiene en seco el actuador apenas detecta el más mínimo contacto físico.
## 1. Principio de Funcionamiento
1. **Aproximación Controlada**: Se configura el dedo seleccionado a una velocidad reducida (`150` en el registro `1522`) para minimizar la inercia mecánica y otorgar tiempo de reacción al bucle de control en Python, manteniendo el techo de fuerza alto (`1000` en el registro `1498`) para vencer la fricción interna de los tendones en vacío.
2. **Lectura con Corrección de Complemento a Dos**: Se consultan los registros de posición real (`1546`) y fuerza real (`1582`) cada $0.02\text{ s}$. Para eliminar el desbordamiento negativo de enteros de $16\text{ bits}$ sin signo (donde pequeñas oscilaciones bajo cero se reportan como $65535$), se aplica la conversión $v - 65536$ para todo valor $v \ge 32768$ y se truncan los valores negativos a $0$.
3. **Freno de Mano por Software**: En el instante en que la lectura de fuerza iguala o supera el umbral asignado ($F_{\text{actual}} \ge U_{\text{efectivo}}$), el script captura el ángulo físico instantáneo del dedo y lo inyecta inmediatamente como la nueva consigna de posición en el registro `1486`, congelando el movimiento en ese punto exacto.2.
## 2. Segmentación de Umbrales por Anatomía del Actuador
Debido a las diferencias mecánicas de fricción interna y masa en cada articulación, el algoritmo discrimina tres niveles de sensibilidad independientes en la función buscar_contacto:
|Índices |DOF|Actuadores Correspondientes|Parámetro en Código|Valor Configurado|Justificación Mecánica|
|---|---|---|---|---|---|
|`0, 1, 2, 3`|Meñique, Anular, Corazón, Índice|`umbral_contacto`|`10`|Esfuerzo basal nulo en vacío; permite máxima sensibilidad al tacto en las yemas.|
|`4`|Flexión pulgar|`umbral_flexion`|`50`|Compensa la mayor tensión del resorte principal del pulgar durante el recorrido libre.|
|`5`|Rotación pulgar|`umbral_rotacion`|`150`|Filtra el offset mecánico propio del riel de oposición del pulgar.|

## 3. Mapa de Registros Modbus Utilizados
- `REG_ANGLE_SET` (`1486`): Escritura directa de la consigna de posición angular ($0 - 1000$).
- `REG_FORCE_SET` (`1498`): Escritura del límite de fuerza operativa ($0 - 1000$).
- `REG_SPEED_SET` (`1522`): Escritura de la velocidad de desplazamiento ($0 - 1000$).
- `REG_ANGLE_ACT` (`1546`): Lectura en tiempo real de los sensores de ángulo actual.
- `REG_FORCE_ACT` (`1582`): Lectura en tiempo real de los sensores de fuerza actual.

## Código Fuente

In [1]:
import time
import sys
import select
from inspire_rh56dftp import InspireHand

IP_MANO = "192.168.123.211"
NOMBRES = ["Meñique", "Anular", "Corazón", "Índice", "Flexión pulgar", "Rotación pulgar"]

REG_ANGLE_SET = 1486 # Consigna de ángulo
REG_FORCE_SET = 1498 # Límite de fuerza configurada
REG_SPEED_SET = 1522 # Velocidad configurada
REG_ANGLE_ACT = 1546 # Sensor de ángulo real
REG_FORCE_ACT = 1582 # Sensor de fuerza real

ABIERTO = [1000, 1000, 1000, 1000, 1000, 1000]
FUERZA_MAXIMA = [1000, 1000, 1000, 1000, 1000, 1000]

def verificar_salida_teclado():
    """Permite romper el bucle presionando 'q' + ENTER en consola."""
    if sys.stdin in select.select([sys.stdin], [], [], 0)[0]:
        entrada = sys.stdin.readline().strip().lower()
        if entrada == 'q':
            return True
    return False

def leer_sensores_rapido(mano):
    """
    Lee posición y fuerza real, aplicando la corrección de complemento a dos 
    para limpiar el ruido negativo del sensor (wrap-around a 65535).
    """
    try:
        angulos_raw = mano.read_holding(REG_ANGLE_ACT, 6)
        angulos = [v if v < 32768 else v - 65536 for v in angulos_raw]
        
        fuerzas_raw = mano.read_holding(REG_FORCE_ACT, 6)
        fuerzas = [max(0, v if v < 32768 else v - 65536) for v in fuerzas_raw]
        
        return angulos, fuerzas
    except Exception:
        return None, None

def buscar_contacto(mano, dedo_idx, umbral_contacto=10, umbral_flexion=50, umbral_rotacion=150):
    """
    Cierra el dedo y lo frena al detectar una fuerza mayor al umbral.
    Aplica 3 niveles de sensibilidad distintos según la anatomía del motor.
    """
    # Discriminación del umbral efectivo según el motor seleccionado
    if dedo_idx == 5:
        umbral_efectivo = umbral_rotacion
    elif dedo_idx == 4:
        umbral_efectivo = umbral_flexion
    else:
        umbral_efectivo = umbral_contacto

    print(f"\n{'='*55}")
    print(f"🔍 MODO PRESENCIA ACTIVO: {NOMBRES[dedo_idx]}")
    print(f">> Umbral de sensibilidad aplicado: {umbral_efectivo}")
    print(f">> El dedo se cerrará lentamente. Toca la yema para detenerlo.")
    print(">> Escribe 'q' y presiona ENTER para abortar, o usa Ctrl+C.")
    print(f"{'='*55}\n")

    # Configuración de aproximación lenta
    velocidad_aproximacion = ABIERTO.copy()
    velocidad_aproximacion[dedo_idx] = 150
    mano._write_and_ack(REG_SPEED_SET, velocidad_aproximacion)
    mano._write_and_ack(REG_FORCE_SET, FUERZA_MAXIMA) 
    
    # Ordenamos al dedo que se cierre por completo
    meta_cierre = ABIERTO.copy()
    meta_cierre[dedo_idx] = 0
    mano._write_and_ack(REG_ANGLE_SET, meta_cierre)

    try:
        while True:
            angulos, fuerzas = leer_sensores_rapido(mano)
            
            if angulos and fuerzas:
                fuerza_actual = fuerzas[dedo_idx]
                angulo_actual = angulos[dedo_idx]
                
                # Telemetría en vivo
                sys.stdout.write(f"\r[Sensores] Ángulo: {angulo_actual:4d} | Fuerza: {fuerza_actual:4d} ")
                sys.stdout.flush()

                # CONDICIÓN DE FRENO: Resistencia mayor al umbral efectivo
                if fuerza_actual >= umbral_efectivo:
                    print(f"\n\n🛑 ¡CONTACTO DETECTADO! Fuerza de {fuerza_actual} superó el umbral de {umbral_efectivo}.")
                    
                    # Freno de mano
                    meta_freno = ABIERTO.copy()
                    meta_freno[dedo_idx] = angulo_actual
                    mano._write_and_ack(REG_ANGLE_SET, meta_freno)
                    print(f"✓ Dedo bloqueado en el ángulo {angulo_actual}.")
                    break
                
                # Si llegó al final del recorrido sin tocar nada
                if angulo_actual <= 20:
                    print("\n\n✓ Fin del recorrido. No se detectó ningún objeto.")
                    break

            if verificar_salida_teclado():
                print("\n\n✓ Salida solicitada por teclado ('q').")
                break

            time.sleep(0.02)
            
    except KeyboardInterrupt:
        print("\n\n✓ Interrupción detectada (Ctrl+C).")
    
    finally:
        print("Liberando el dedo...")
        mano._write_and_ack(REG_SPEED_SET, [1000]*6)
        mano._write_and_ack(REG_ANGLE_SET, ABIERTO)
        time.sleep(1)

def main():
    print(f"Conectando a la mano en {IP_MANO} ...")
    mano = InspireHand(ip=IP_MANO)

    try:
        mano.send_raw(ABIERTO)
        time.sleep(1)
        
        while True:
            print("\n--- MENÚ DE SENSOR DE PRESENCIA ---")
            for i, nombre in enumerate(NOMBRES):
                print(f"{i}. Probar con {nombre}")
            print("-1. Salir")
            
            try:
                sys.stdout.flush()
                time.sleep(0.1)
                sel = int(input("\nSeleccione el dedo a probar (-1 para salir): "))
                
                if sel == -1:
                    break
                    
                if 0 <= sel <= 5:
                    # Inyección de los 3 niveles de sensibilidad
                    buscar_contacto(mano, sel, umbral_contacto=10, umbral_flexion=50, umbral_rotacion=150)
                else:
                    print("⚠ Selección fuera de rango.")
            except ValueError:
                print("⚠ Entrada no válida.")

    except KeyboardInterrupt:
        print("\nCerrando sistema...")
    finally:
        mano._write_and_ack(REG_SPEED_SET, [1000]*6)
        mano.send_raw(ABIERTO)
        mano.close()
        print("Conexión TCP cerrada.")

if __name__ == "__main__":
    main()

Conectando a la mano en 192.168.123.211 ...

--- MENÚ DE SENSOR DE PRESENCIA ---
0. Probar con Meñique
1. Probar con Anular
2. Probar con Corazón
3. Probar con Índice
4. Probar con Flexión pulgar
5. Probar con Rotación pulgar
-1. Salir



Seleccione el dedo a probar (-1 para salir):  4



🔍 MODO PRESENCIA ACTIVO: Flexión pulgar
>> Umbral de sensibilidad aplicado: 50
>> El dedo se cerrará lentamente. Toca la yema para detenerlo.
>> Escribe 'q' y presiona ENTER para abortar, o usa Ctrl+C.

[Sensores] Ángulo:    5 | Fuerza:   21 

✓ Fin del recorrido. No se detectó ningún objeto.
Liberando el dedo...

--- MENÚ DE SENSOR DE PRESENCIA ---
0. Probar con Meñique
1. Probar con Anular
2. Probar con Corazón
3. Probar con Índice
4. Probar con Flexión pulgar
5. Probar con Rotación pulgar
-1. Salir



Seleccione el dedo a probar (-1 para salir):  4



🔍 MODO PRESENCIA ACTIVO: Flexión pulgar
>> Umbral de sensibilidad aplicado: 50
>> El dedo se cerrará lentamente. Toca la yema para detenerlo.
>> Escribe 'q' y presiona ENTER para abortar, o usa Ctrl+C.

[Sensores] Ángulo:    0 | Fuerza:   18 

✓ Fin del recorrido. No se detectó ningún objeto.
Liberando el dedo...

--- MENÚ DE SENSOR DE PRESENCIA ---
0. Probar con Meñique
1. Probar con Anular
2. Probar con Corazón
3. Probar con Índice
4. Probar con Flexión pulgar
5. Probar con Rotación pulgar
-1. Salir


...


🔍 MODO PRESENCIA ACTIVO: Flexión pulgar
>> Umbral de sensibilidad aplicado: 50
>> El dedo se cerrará lentamente. Toca la yema para detenerlo.
>> Escribe 'q' y presiona ENTER para abortar, o usa Ctrl+C.

[Sensores] Ángulo:    5 | Fuerza:   18 

✓ Fin del recorrido. No se detectó ningún objeto.
Liberando el dedo...

--- MENÚ DE SENSOR DE PRESENCIA ---
0. Probar con Meñique
1. Probar con Anular
2. Probar con Corazón
3. Probar con Índice
4. Probar con Flexión pulgar
5. Probar con Rotación pulgar
-1. Salir



Seleccione el dedo a probar (-1 para salir):  4



🔍 MODO PRESENCIA ACTIVO: Flexión pulgar
>> Umbral de sensibilidad aplicado: 50
>> El dedo se cerrará lentamente. Toca la yema para detenerlo.
>> Escribe 'q' y presiona ENTER para abortar, o usa Ctrl+C.

[Sensores] Ángulo:    5 | Fuerza:   20 

✓ Fin del recorrido. No se detectó ningún objeto.
Liberando el dedo...

--- MENÚ DE SENSOR DE PRESENCIA ---
0. Probar con Meñique
1. Probar con Anular
2. Probar con Corazón
3. Probar con Índice
4. Probar con Flexión pulgar
5. Probar con Rotación pulgar
-1. Salir



Seleccione el dedo a probar (-1 para salir):  -1


Conexión TCP cerrada.
